Cell 1 — Install:

In [17]:
!pip install scikit-learn datasets -q
print("✅ Done!")

✅ Done!


Cell 2 — Load data:

In [18]:
from datasets import load_dataset
import pandas as pd
from sklearn.model_selection import train_test_split

dataset = load_dataset("tweet_eval", "hate")

# Combine ALL splits into one
all_df = pd.concat([
    pd.DataFrame(dataset["train"]),
    pd.DataFrame(dataset["validation"]),
    pd.DataFrame(dataset["test"])
]).reset_index(drop=True)

print(f"Total rows: {len(all_df)}")
print(f"Label counts:\n{all_df['label'].value_counts()}")

# Split ourselves — 80% train, 20% test (same distribution guaranteed)
train_df, test_df = train_test_split(
    all_df, test_size=0.2,
    random_state=42,
    stratify=all_df["label"]  # ensures same ratio in both splits
)

X_train = train_df["text"].tolist()
y_train = train_df["label"].tolist()
X_test  = test_df["text"].tolist()
y_test  = test_df["label"].tolist()

print(f"\nTrain: {len(X_train)} | Test: {len(X_test)}")
print(f"Train label 0: {y_train.count(0)} | label 1: {y_train.count(1)}")
print(f"Test  label 0: {y_test.count(0)}  | label 1: {y_test.count(1)}")
print("✅ Data ready!")

Total rows: 12970
Label counts:
label
0    7508
1    5462
Name: count, dtype: int64

Train: 10376 | Test: 2594
Train label 0: 6006 | label 1: 4370
Test  label 0: 1502  | label 1: 1092
✅ Data ready!


Cell 3 — Train model:

In [19]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report

tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1,2))
X_train_vec = tfidf.fit_transform(X_train)
X_test_vec  = tfidf.transform(X_test)

clf = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
clf.fit(X_train_vec, y_train)

preds = clf.predict(X_test_vec)
print(f"Prediction counts — 0: {list(preds).count(0)} | 1: {list(preds).count(1)}")
print(f"Actual counts     — 0: {y_test.count(0)}  | 1: {y_test.count(1)}")

acc = accuracy_score(y_test, preds)
f1  = f1_score(y_test, preds, average="weighted")

print("\n" + classification_report(
    y_test, preds,
    target_names=["Not Hate", "Hate Speech"]
))
print(f"✅ Accuracy : {acc:.4f}")
print(f"✅ F1 Score : {f1:.4f}")

Prediction counts — 0: 1639 | 1: 955
Actual counts     — 0: 1502  | 1: 1092

              precision    recall  f1-score   support

    Not Hate       0.75      0.82      0.78      1502
 Hate Speech       0.71      0.62      0.67      1092

    accuracy                           0.74      2594
   macro avg       0.73      0.72      0.72      2594
weighted avg       0.73      0.74      0.73      2594

✅ Accuracy : 0.7359
✅ F1 Score : 0.7329


Cell 4 — Log to MLflow:

In [20]:
import mlflow
import os

os.environ["MLFLOW_TRACKING_USERNAME"] = "harshmemane"
os.environ["MLFLOW_TRACKING_PASSWORD"] = "5291bd2752ef80317b480b80cefc12b281b0732a"

mlflow.set_tracking_uri(
    "https://dagshub.com/harshmemane/hate-speech-detection.mlflow"
)
mlflow.set_experiment("hate-speech-detection")

with mlflow.start_run():
    mlflow.log_param("model",        "LogisticRegression")
    mlflow.log_param("vectorizer",   "TF-IDF")
    mlflow.log_param("max_features", 10000)
    mlflow.log_param("ngram_range",  "(1,2)")
    mlflow.log_metric("accuracy",    acc)
    mlflow.log_metric("f1_score",    f1)
    mlflow.sklearn.log_model(tfidf, "tfidf-vectorizer")
    mlflow.sklearn.log_model(clf,   "classifier")
    print("✅ Logged to MLflow!")

2026/05/03 15:14:02 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/03 15:14:05 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/05/03 15:14:05 WARNING mlflow.sklearn: Model was missing function: predict. Not logging python_function flavor!
2026/05/03 15:14:12 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/03 15:14:19 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The r

✅ Logged to MLflow!
🏃 View run sneaky-grub-373 at: https://dagshub.com/harshmemane/hate-speech-detection.mlflow/#/experiments/0/runs/0a75b0d31cec42afa84489093cdbda25
🧪 View experiment at: https://dagshub.com/harshmemane/hate-speech-detection.mlflow/#/experiments/0


Cell 5 — Save model:

In [21]:
import pickle

with open("hate_speech_model.pkl", "wb") as f:
    pickle.dump({"tfidf": tfidf, "clf": clf}, f)

print("✅ Model saved as hate_speech_model.pkl")

✅ Model saved as hate_speech_model.pkl


Cell 6 — Test it:

In [23]:
test_sentences = [
    "Have a wonderful day everyone!",
    "kill all people from that religion they dont deserve to live",
    "I disagree with your political opinion",
    "those immigrants are destroying our country get them out",
    "great game today, well played!"
]

label_map = {0: "Not Hate", 1: "Hate Speech"}

print("=== Model Predictions ===")
for s in test_sentences:
    vec  = tfidf.transform([s])
    pred = clf.predict(vec)[0]
    prob = clf.predict_proba(vec)[0][pred]
    print(f"  '{s}'")
    print(f"  → {label_map[pred]} ({prob*100:.1f}% confidence)\n")

=== Model Predictions ===
  'Have a wonderful day everyone!'
  → Not Hate (67.5% confidence)

  'kill all people from that religion they dont deserve to live'
  → Not Hate (52.1% confidence)

  'I disagree with your political opinion'
  → Not Hate (71.2% confidence)

  'those immigrants are destroying our country get them out'
  → Hate Speech (86.7% confidence)

  'great game today, well played!'
  → Not Hate (79.4% confidence)



In [24]:
from google.colab import files
files.download("hate_speech_model.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>